# 🚀 FASE 0: Preprocessing Unificato e Intelligente

Questo notebook va ad agire in modo **sicuro** sui tuoi dati grezzi (Raw). Niente percorsi scritti a mano: cercherà automaticamente i 6 dataset nelle cartelle che alleghi (ADARP, SPD, WEEE, WESD, ue4w, BIG-IDEAS) e produrrà un archivio pre-elaborato inattaccabile.

In [ ]:
import os, sys, shutil, time

WORK_DIR = '/kaggle/working/e4selflearning'
os.chdir('/kaggle/working')

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)

print("🚀 Clonazione repository...")
!git clone https://github.com/april-tools/e4selflearning.git {WORK_DIR}

print("📦 Installazione dipendenze...")
!pip uninstall -y -q numpy scipy pandas neurokit2 biopsykit
!pip install --no-cache-dir -q "numpy==1.26.4" "scipy==1.12.0" "pandas<2.2" biopsykit flirt ruamel.yaml biosppy hrv-analysis neurokit2 distinctipy joypy peakutils lightning torchmetrics

print("\n⚠️ RIAVVIA IL KERNEL ORA (Menu: Run -> Restart Kernel) e poi prosegui dalla Cella 2.")
import os
os._exit(0)

In [ ]:
import os, sys, glob
sys.path.insert(0, '/kaggle/working/e4selflearning/src')
from timebase.data.static import UNLABELLED_DATA_PATHS

os.chdir('/kaggle/working/e4selflearning')
UNLABELLED_DIR = '/kaggle/working/e4selflearning/data/raw_data/unlabelled_data'
os.makedirs(UNLABELLED_DIR, exist_ok=True)

print("🔍 Ricerca automatica dei dataset grezzi...")
target_datasets = {
    'ADARP': 'ADARP/Data/Data',
    'SPD': 'SPD/Raw_data/Raw_data',
    'WEEE': 'WEEE/dataset/dataset',
    'WESD': 'WESD/Data/Data',
    'ue4w': 'ue4w',
    'BIG-IDEAS': 'big-ideas-lab-glycemic'
}

found_paths = {}
all_dirs = [x[0] for x in os.walk('/kaggle/input')]

for name, partial_path in target_datasets.items():
    for d in all_dirs:
        if partial_path in d:
            if name == 'ue4w' and 'WEEE' in d: continue
            if name == 'BIG-IDEAS':
                # cerchiamo la cartella che contiente 001, 002 ecc
                if os.path.isdir(os.path.join(d, '001')):
                    found_paths[name] = d
                    break
            else:
                found_paths[name] = d
                break

for name, src in found_paths.items():
    coll_key = name.lower() if name != 'BIG-IDEAS' else 'big-ideas'
    dst = os.path.join(UNLABELLED_DIR, UNLABELLED_DATA_PATHS.get(coll_key, name))
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not os.path.exists(dst) and os.path.exists(src):
        os.symlink(src, dst)
        print(f"✅ {name} TROVATO e collegato -> {src}")

missing = set(target_datasets.keys()) - set(found_paths.keys())
if missing:
    print(f"\n⚠️ ATTENZIONE: Non sono riuscita a trovare questi dataset nell'input: {missing}")
else:
    print("\n🎉 TUTTI E 6 I DATASET SONO STATI TROVATI!")

In [ ]:
import os, sys, pickle, argparse, gc, shutil
import numpy as np
os.chdir('/kaggle/working/e4selflearning')
sys.path.insert(0, 'src')
from timebase.data import preprocessing as prep
from timebase.data.static import LABEL_COLS, UNLABELLED_DATA_PATHS

# Patches per bug conosciuti
!sed -i 's/os.walk(root_dir)/os.walk(root_dir, followlinks=True)/g' src/timebase/data/preprocessing.py
!sed -i 's/header=None,).values/header=None, low_memory=False).values; try: float(raw_data[0,0]); except: raw_data = raw_data[1:]/g' src/timebase/data/preprocessing.py

BASE = 'data/preprocessed/unsegmented_unlabelled'
os.makedirs(BASE, exist_ok=True)
meta_p = os.path.join(BASE, 'metadata.pkl')
combined_meta = {'invalid_sessions':[], 'sessions_info':{}, 'sleep_algorithm':'van_hees'}

args = argparse.Namespace(path2unlabelled_data='data/raw_data/unlabelled_data', output_dir=BASE, e4selflearning=True, overwrite=True, sleep_algorithm='van_hees', wear_minimum_minutes=5, minimum_recorded_time=15, num_workers=2, chunksize=1, verbose=0, seed=1234)

print("⏳ INIZIO PRE-ELABORAZIONE LUNGA (Mettiti comoda, ci vorranno ~2 ore)...")
datasets_to_process = [k.lower() if k != 'BIG-IDEAS' else 'big-ideas' for k in found_paths.keys()]

for col in datasets_to_process:
    print(f"\n🚀 Elaborazione {col}...")
    s_dirs = prep.recast_collection(args, collection=col, path=UNLABELLED_DATA_PATHS[col])
    for s_id in s_dirs:
        name = s_id.replace('data/raw_data/unlabelled_data/recast/', '')
        if name in combined_meta['sessions_info']: continue
        try:
            data, info, too_short = prep.preprocess_dir(args, recording_dir=s_id, labelled=False)
            if not too_short:
                data['labels'] = np.full(len(LABEL_COLS), np.nan, dtype=np.float32)
                os.makedirs(os.path.join(BASE, name), exist_ok=True)
                from timebase.utils import h5
                # compressione opzionale che salva spazio e IO
                h5.write(os.path.join(BASE, name, 'channels.h5'), data, True)
                combined_meta['sessions_info'][name] = info
                with open(meta_p, 'wb') as f: pickle.dump(combined_meta, f)
            del data; gc.collect()
        except Exception as e: print(f"  ⚠️ Errore {name}")
    shutil.rmtree(os.path.join(args.path2unlabelled_data, 'recast', col), ignore_errors=True)

print("\n✅ PREPROCESSING FINITO CON SUCCESSO!")

In [ ]:
import shutil
print("📦 Creazione dell'archivio zippato finale...")
shutil.make_archive('DatiPreelaboratiFinale', 'gztar', '/kaggle/working/e4selflearning/data/preprocessed/unsegmented_unlabelled')
print("✅ Archivio pronto! Scarica 'DatiPreelaboratiFinale.tar.gz' e caricalo come Dataset su Kaggle.")